# Boundary File EDA — UK Local Authority Boundaries

This notebook investigates the structure, columns and contents of uk_local_authority_boundaries.geojson, in preparation for Version 2's AOI clipping feature — cropping the Sentinel-2 image to the Stoke-on-Trent boundary specifically, and supporting geographic validation.

## Objectives
- Confirm the file loads correctly and check its overall structure.
- Identify how many areas the file contains, and what fields/columns describe each one.
- Confirm the GSS code field exists and can be used to filter for Stoke-on-Trent specifically (E06000021, as documented in data/README.md).
- Check the coordinate format used for the boundary geometry, and whether it's compatible with — or convertible to — the satellite image's coordinate system (EPSG:32630).

In [17]:
import os
from pathlib import Path

import ijson

In [18]:
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path(os.getcwd()).parent
geojson_path = str(PROJECT_ROOT / "data" / "uk_local_authority_boundaries.geojson")
print(geojson_path)

C:\Users\lward\workspace\sentinel2-brownfield-stoke\data\uk_local_authority_boundaries.geojson


In [19]:
with open(geojson_path, 'rb') as f:
    raw_data = f.read()
    print(len(raw_data))
    print(raw_data[:2000])

375842305
b'{"type":"FeatureCollection","crs":{"type":"name","properties":{"name":"EPSG:4326"}},"features":[{"type":"Feature","id":1,"geometry":{"type":"Polygon","coordinates":[[[-1.26845558516825,54.7261163511949],[-1.26822471737476,54.7260906130628],[-1.26814840501847,54.7261055160614],[-1.26799879308498,54.7262169172337],[-1.26795305492628,54.7262615617691],[-1.26779129849886,54.7267062911412],[-1.26774209936712,54.7267653797089],[-1.26767139683588,54.7268098743903],[-1.26763078457252,54.7268258010578],[-1.26752161954628,54.7268520946407],[-1.26740488925401,54.7268648668595],[-1.26727145724849,54.7268595700975],[-1.26660520485977,54.7267827756806],[-1.2661828460771,54.7266983781983],[-1.26589678017128,54.726631064916],[-1.26569842708319,54.7265202592553],[-1.26502938316548,54.7260812900198],[-1.26496633407424,54.7260493741588],[-1.26488776157024,54.7260192515171],[-1.26480110816502,54.725997165236],[-1.26470984439937,54.7259822376381],[-1.26458416117142,54.7259733913404],[-1.2644460

## Finding — File Structure and Coordinate System

The UK Local Authority Boundaries file is approximately 358MB, which exceeds GitHub's fixed 100MB-per-file limit. For this reason it is excluded from the repository via .gitignore, with download instructions documented instead in data/README.md.
Raw byte inspection confirms this file genuinely is plain-text JSON, unlike the brownfield register file investigated in 02_brownfield_register_eda.ipynb, which was disguised as a CSV. The file also explicitly states its own coordinate system within the data itself (EPSG:4326), the same standard latitude/longitude system used when converting the satellite tile bounds in 01_data_inspection.ipynb — meaning no inference is needed here, unlike the register's coordinate system.

In [16]:
stoke = raw_data.find(b'E06000021')
print(stoke)
print(raw_data[10788659:10789100])

10788659
b'E06000021","LAD24NM":"Stoke-on-Trent","LAD24NMW":" ","BNG_E":389438,"BNG_N":346650,"LONG":-2.15888,"LAT":53.0171}},{"type":"Feature","id":22,"geometry":{"type":"Polygon","coordinates":[[[-2.34289724784043,51.4393863205059],[-2.34275048557933,51.439383154501],[-2.34236485736425,51.4393860803923],[-2.34225721019409,51.4394106665911],[-2.34217229080809,51.4394127122875],[-2.34208747453718,51.4394129601707],[-2.34203698692447,51.439403219471'


In [23]:
with open(geojson_path, 'rb') as f:
    for feature in ijson.items(f, 'features.item'):
        if feature['properties']['LAD24CD'] == 'E06000021':
            stoke_feature = feature
            break
print(stoke_feature)

{'type': 'Feature', 'id': 21, 'geometry': {'type': 'Polygon', 'coordinates': [[[Decimal('-2.19595385726922'), Decimal('53.09173668887')], [Decimal('-2.19589612353286'), Decimal('53.09173589466')], [Decimal('-2.19572149793959'), Decimal('53.09175235967')], [Decimal('-2.19566019416776'), Decimal('53.0917542580705')], [Decimal('-2.19559751725701'), Decimal('53.0917516660071')], [Decimal('-2.19545247005306'), Decimal('53.0917204534292')], [Decimal('-2.19530470401782'), Decimal('53.0917144071895')], [Decimal('-2.19520019143829'), Decimal('53.0917172756458')], [Decimal('-2.19515682773715'), Decimal('53.0917110567512')], [Decimal('-2.19511344230033'), Decimal('53.0917003442744')], [Decimal('-2.19506107588449'), Decimal('53.0916635865871')], [Decimal('-2.1948857377856'), Decimal('53.0915218001647')], [Decimal('-2.19473900342324'), Decimal('53.0914438601962')], [Decimal('-2.19462095106143'), Decimal('53.0914197915513')], [Decimal('-2.19455371902021'), Decimal('53.0914172064495')], [Decimal('-2.

## Finding — Stoke-on-Trent Feature Located via ijson

Using ijson to iterate through the file's features one at a time, the feature matching GSS code E06000021 was successfully located and confirmed as Stoke-on-Trent (LAD24NM: Stoke-on-Trent). This approach avoided loading the full 358MB file into memory, and avoided the earlier risk of accidentally pulling a neighbouring feature's geometry (as nearly happened with Hartlepool's adjacent feature when searching by raw byte position).

Stoke's properties give BNG_E: 389438, BNG_N: 346650 — closely matching the brownfield register's own site coordinates investigated in 02_brownfield_register_eda.ipynb (e.g. GeoX: 388309.18, GeoY: 344107.34). This is strong supporting evidence that the inferred British National Grid coordinate system for the register was correct, since both datasets independently sit in the same numeric range for the same geographic area.

The properties also provide LONG: -2.15888 and LAT: 53.0171 directly, meaning the coordinates were ready to use without needing extraction work via the polygon's nested arrays — though, like the boundary outline itself, these values will still need converting to EPSG:32630 before use in Version 2's AOI clipping.

In [28]:
stoke_boundary = feature['geometry']['coordinates'][0]
print(stoke_boundary)

[[Decimal('-2.19595385726922'), Decimal('53.09173668887')], [Decimal('-2.19589612353286'), Decimal('53.09173589466')], [Decimal('-2.19572149793959'), Decimal('53.09175235967')], [Decimal('-2.19566019416776'), Decimal('53.0917542580705')], [Decimal('-2.19559751725701'), Decimal('53.0917516660071')], [Decimal('-2.19545247005306'), Decimal('53.0917204534292')], [Decimal('-2.19530470401782'), Decimal('53.0917144071895')], [Decimal('-2.19520019143829'), Decimal('53.0917172756458')], [Decimal('-2.19515682773715'), Decimal('53.0917110567512')], [Decimal('-2.19511344230033'), Decimal('53.0917003442744')], [Decimal('-2.19506107588449'), Decimal('53.0916635865871')], [Decimal('-2.1948857377856'), Decimal('53.0915218001647')], [Decimal('-2.19473900342324'), Decimal('53.0914438601962')], [Decimal('-2.19462095106143'), Decimal('53.0914197915513')], [Decimal('-2.19455371902021'), Decimal('53.0914172064495')], [Decimal('-2.1943163760221'), Decimal('53.091422089641')], [Decimal('-2.19414018419887'), D

## Finding — Stoke-on-Trent Boundary Extracted
Stoke-on-Trent's boundary list is encapsulated within the GeoJSON Polygon inner list. Using ijson to capture only the Stoke-on-Trent coordinates and saving them to the variable stoke_boundary. The list contains longitude/latitude pairs that trace the actual boundary line. As stoke contains no different council areas within the boundary (ie the boundary line is unbroken, no holes and/or no enclaves), coordinates contain exactly one ring, at index 0. The CRS for the coordinates is EPSG:4326 as stated earlier. This is the correct format for extraction, but a conversion to EPSG:32630 (matching the Sentinel-2 satellite image) will be required before this boundary can be used for AOI clipping in Version 2.
